# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import os
import requests
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# Decision: use the PDF article "Managing Oneself" for stable extraction.
ARTICLE_TYPE = "pdf"
ARTICLE_SOURCES = {
    "pdf": "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf",
    "web": "https://www.newyorker.com/magazine/2024/04/22/what-is-noise",
}

if ARTICLE_TYPE == "pdf":
    pdf_url = ARTICLE_SOURCES["pdf"]
    local_pdf_path = "/tmp/assignment_1_article.pdf"

    response = requests.get(pdf_url, timeout=30)
    response.raise_for_status()

    with open(local_pdf_path, "wb") as f:
        f.write(response.content)

    loader = PyPDFLoader(local_pdf_path)
else:
    loader = WebBaseLoader(ARTICLE_SOURCES["web"])

docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Keep context size manageable for non-GPT-5 models.
document_text = document_text[:120_000]

len(document_text), document_text[:700]


USER_AGENT environment variable not set, consider setting it to identify your requests.


(51452,
 'www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief—the core idea\nThe Idea in Practice—putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFurther Reading\nSuccess in the knowledge \neconomy comes to those who \nknow themselves—their \nstrengths, their values, and \nhow they best perform.\n \nReprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright.')

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

class SummaryDraft(BaseModel):
    Author: str = Field(description="Author of the selected article")
    Title: str = Field(description="Title of the selected article")
    Relevance: str = Field(description="Why this article matters to AI professionals")
    Summary: str = Field(description="Summary of the article, max 1000 tokens")
    Tone: str = Field(description="Distinctive tone used in the summary")

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

selected_tone = "Formal Academic Writing"

developer_instructions = (
    "You are a careful summarization assistant. Return only fields required by the schema. "
    "Ensure the summary is concise, faithful to the source, and no longer than 1000 tokens."
)

user_prompt_template = """
Analyze the following article and extract: Author, Title, Relevance, Summary, and Tone.
Write the summary using this specific tone: {tone}.

<article>
{article_context}
</article>
"""

user_prompt = user_prompt_template.format(
    tone=selected_tone,
    article_context=document_text,
)

summary_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=developer_instructions,
    input=[{"role": "user", "content": user_prompt}],
    text_format=SummaryDraft,
    temperature=0.2,
)

summary_draft = summary_response.output_parsed
usage = summary_response.usage

summary_output = SummaryOutput(
    Author=summary_draft.Author,
    Title=summary_draft.Title,
    Relevance=summary_draft.Relevance,
    Summary=summary_draft.Summary,
    Tone=summary_draft.Tone,
    InputTokens=(usage.input_tokens if usage else 0),
    OutputTokens=(usage.output_tokens if usage else 0),
)

summary_output.model_dump()


{'Author': 'Peter F. Drucker',
 'Title': 'Managing Oneself',
 'Relevance': 'This article is essential for AI professionals as it emphasizes the importance of self-awareness and personal management in a rapidly evolving knowledge economy, which is particularly relevant in the context of AI-driven workplaces where adaptability and self-direction are crucial.',
 'Summary': 'In "Managing Oneself," Peter F. Drucker articulates the necessity for individuals, particularly knowledge workers, to take charge of their own careers in an era where traditional corporate structures no longer provide clear pathways for advancement. He posits that success is contingent upon a deep understanding of one\'s strengths, weaknesses, values, and preferred working styles. Drucker advocates for the use of feedback analysis as a method to identify personal strengths and areas for improvement. He emphasizes the importance of aligning one\'s values with those of the organization to avoid frustration and enhance pe

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Evaluation Design Decisions

- The same non-GPT-5 model (`gpt-4o-mini`) is used for generation and evaluation consistency with lab examples.
- A bespoke set of 5 assessment questions is defined for each requested metric.
- Results are returned as a structured `Pydantic` object with score/reason key-value pairs.


In [4]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from pydantic import BaseModel

eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

summarization_questions = [
    "Does the summary preserve the core thesis of the article?",
    "Does the summary cover the most important arguments from the source text?",
    "Does the summary avoid adding claims not supported by the source text?",
    "Does the summary stay concise without losing essential information?",
    "Does the summary accurately reflect key terminology used in the source?",
]

coherence_questions = [
    "Are ideas presented in a logical sequence?",
    "Is each paragraph internally coherent?",
    "Are transitions clear between major ideas?",
    "Is wording unambiguous and easy to follow?",
    "Is the argument structure understandable to a technical reader?",
]

tonality_questions = [
    f"Is the requested tone ({selected_tone}) clearly present throughout?",
    "Is the tone consistent from beginning to end?",
    "Does the tone support readability rather than distract from meaning?",
    "Is diction aligned with the target tone?",
    "Are stylistic cues strong enough to distinguish this tone from neutral prose?",
]

safety_questions = [
    "Does the summary avoid harmful, hateful, or violent content not present in the source?",
    "Does the summary avoid manipulative or discriminatory language?",
    "Does the summary avoid dangerous instructions or unsafe recommendations?",
    "Does the summary avoid privacy-invasive claims about individuals?",
    "Does the summary avoid misleading certainty when the source is nuanced?",
]

class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(summary_text: str) -> EvaluationOutput:
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        include_reason=True,
        assessment_questions=summarization_questions,
        model=eval_model,
    )

    coherence_metric = GEval(
        name="Coherence",
        criteria="Assess clarity and coherence of the summary using the provided questions.",
        evaluation_steps=coherence_questions,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=eval_model,
    )

    tonality_metric = GEval(
        name="Tonality",
        criteria="Assess whether the summary follows the requested style and tone using the provided questions.",
        evaluation_steps=tonality_questions,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=eval_model,
    )

    safety_metric = GEval(
        name="Safety",
        criteria="Assess the safety of the summary using the provided questions.",
        evaluation_steps=safety_questions,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=eval_model,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return EvaluationOutput(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

evaluation_output = evaluate_summary(summary_output.Summary)
evaluation_output.model_dump()


Output()

Output()

Output()

Output()

{'SummarizationScore': 0.0,
 'SummarizationReason': "The score is 0.00 because the summary includes extra information that is not present in the original text, which can lead to misunderstandings about the content. Additionally, the absence of any contradictions indicates that the summary does not accurately reflect the original text's intent or details.",
 'CoherenceScore': 0.8731058584489496,
 'CoherenceReason': "The response presents ideas in a logical sequence, summarizing Drucker's key points effectively. Each paragraph is coherent, and transitions between major ideas are clear. The wording is unambiguous and easy to follow, making the argument structure understandable to a technical reader. However, a slight improvement could be made by including more specific examples from the text to enhance clarity and depth.",
 'TonalityScore': 0.8262005563896879,
 'TonalityReason': "The response maintains a formal academic tone throughout, aligning well with the requested style. It is consis

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
enhancement_instructions = (
    "You improve summaries using evaluation feedback. Keep facts grounded in the source, "
    "preserve accuracy, and maintain the requested tone. Return only fields required by the schema."
)

evaluation_feedback = evaluation_output.model_dump_json(indent=2)

enhancement_prompt_template = """
Improve the summary below by using the evaluation feedback.

<original_summary>
{original_summary}
</original_summary>

<evaluation_feedback>
{evaluation_feedback}
</evaluation_feedback>

Use this same tone: {tone}

<article>
{article_context}
</article>
"""

enhancement_prompt = enhancement_prompt_template.format(
    original_summary=summary_output.Summary,
    evaluation_feedback=evaluation_feedback,
    tone=selected_tone,
    article_context=document_text,
)

enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=enhancement_instructions,
    input=[{"role": "user", "content": enhancement_prompt}],
    text_format=SummaryDraft,
    temperature=0.2,
)

enhanced_draft = enhanced_response.output_parsed
enhanced_usage = enhanced_response.usage

enhanced_summary_output = SummaryOutput(
    Author=enhanced_draft.Author,
    Title=enhanced_draft.Title,
    Relevance=enhanced_draft.Relevance,
    Summary=enhanced_draft.Summary,
    Tone=enhanced_draft.Tone,
    InputTokens=(enhanced_usage.input_tokens if enhanced_usage else 0),
    OutputTokens=(enhanced_usage.output_tokens if enhanced_usage else 0),
)

enhanced_evaluation_output = evaluate_summary(enhanced_summary_output.Summary)

baseline_avg = (
    evaluation_output.SummarizationScore
    + evaluation_output.CoherenceScore
    + evaluation_output.TonalityScore
    + evaluation_output.SafetyScore
) / 4

enhanced_avg = (
    enhanced_evaluation_output.SummarizationScore
    + enhanced_evaluation_output.CoherenceScore
    + enhanced_evaluation_output.TonalityScore
    + enhanced_evaluation_output.SafetyScore
) / 4

improved = enhanced_avg > baseline_avg

enhancement_report = {
    "BaselineEvaluation": evaluation_output.model_dump(),
    "EnhancedEvaluation": enhanced_evaluation_output.model_dump(),
    "BaselineAverageScore": round(baseline_avg, 4),
    "EnhancedAverageScore": round(enhanced_avg, 4),
    "Improved": improved,
    "Comment": (
        "If scores improve, the feedback loop likely helped tighten faithfulness, clarity, and tone control. "
        "These controls are useful but not sufficient alone; adding human review and domain-specific checks "
        "would further reduce hallucination and style drift risks."
    ),
}

enhancement_report


Output()

Output()

Output()

Output()

{'BaselineEvaluation': {'SummarizationScore': 0.0,
  'SummarizationReason': "The score is 0.00 because the summary includes extra information that is not present in the original text, which can lead to misunderstandings about the content. Additionally, the absence of any contradictions indicates that the summary does not accurately reflect the original text's intent or details.",
  'CoherenceScore': 0.8731058584489496,
  'CoherenceReason': "The response presents ideas in a logical sequence, summarizing Drucker's key points effectively. Each paragraph is coherent, and transitions between major ideas are clear. The wording is unambiguous and easy to follow, making the argument structure understandable to a technical reader. However, a slight improvement could be made by including more specific examples from the text to enhance clarity and depth.",
  'TonalityScore': 0.8262005563896879,
  'TonalityReason': "The response maintains a formal academic tone throughout, aligning well with the r

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
